# Module 05 — Notebook 2: Distributions and Scatter Plots

## Learning Objectives

By the end of this notebook you will be able to:
- Create histograms to visualize score distributions
- Create scatter plots to explore relationships between two variables
- Use seaborn for cleaner statistical visualizations with less code
- Create box plots and strip plots to show per-group distributions
- Interpret what each chart type tells you

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json
from pathlib import Path

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

CSV_PATH  = Path("../../data/synthetic/evaluation_results.csv")
JSON_PATH = Path("../../data/synthetic/model_outputs.json")

df = pd.read_csv(CSV_PATH)
with open(JSON_PATH) as f:
    outputs_df = pd.DataFrame(json.load(f))
outputs_df["response_length"] = outputs_df["response"].str.len()

print("seaborn version:", sns.__version__)
print("eval_df:", df.shape, "  outputs_df:", outputs_df.shape)

## 1. Histograms — Understanding Distributions

A histogram groups values into bins and shows how many fall in each bin. It answers:
"What does the distribution of this variable look like?"

In JS you'd need to compute bin counts manually. In Python:
```python
ax.hist(values, bins=10)
```
Or with seaborn (adds a smooth density curve):
```python
sns.histplot(data=df, x="score", kde=True, ax=ax)
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: matplotlib histogram
axes[0].hist(df["score"], bins=8, color="steelblue", edgecolor="white", linewidth=0.8)
axes[0].axvline(df["score"].mean(), color="crimson", linestyle="--",
                label=f"mean={df['score'].mean():.3f}")
axes[0].set_title("Score Distribution (matplotlib)")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Right: seaborn histplot with KDE
sns.histplot(data=df, x="score", kde=True, bins=8, ax=axes[1], color="steelblue")
axes[1].axvline(df["score"].mean(), color="crimson", linestyle="--")
axes[1].set_title("Score Distribution (seaborn + KDE)")
axes[1].set_xlabel("Score")

plt.suptitle("All 20 Evaluation Scores", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Box Plots — Per-Group Distributions

A box plot (box-and-whisker) shows the median, quartiles, and outliers for each group.
It answers: "How do the score distributions compare across models?"

seaborn's `boxplot` takes a whole DataFrame — no manual groupby needed.

```python
sns.boxplot(data=df, x="model", y="score", ax=ax)
```

With only 5 data points per model, also add individual dots with `sns.stripplot`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=df, x="model", y="score",
    palette="muted", width=0.5, ax=ax
)
# Overlay individual points so we can see all 5 scores per model
sns.stripplot(
    data=df, x="model", y="score",
    color="black", size=7, alpha=0.7, jitter=False, ax=ax
)

ax.axhline(df["score"].mean(), color="gray", linestyle="--", linewidth=1,
           label=f"overall mean ({df['score'].mean():.3f})")
ax.set_title("Score Distribution per Model", fontsize=13)
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_ylim(0.5, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Scatter Plots — Relationships Between Variables

A scatter plot puts one variable on x and another on y. Each point is one observation.
It answers: "Is there a relationship between X and Y?"

Here we'll plot each model's v1 score vs. its v2 score per task, to see
whether v2 improved on every task (all points above the y=x diagonal means yes).

In [ ]:
# Build a v1 vs v2 comparison for both model families
def get_v1_v2(family):
    v1 = df[df["model"] == f"{family}-v1"].set_index("task")["score"]
    v2 = df[df["model"] == f"{family}-v2"].set_index("task")["score"]
    return v1, v2

a1, a2 = get_v1_v2("model-a")
b1, b2 = get_v1_v2("model-b")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (v1, v2), label, color in [
    (axes[0], (a1, a2), "model-a", "#2196F3"),
    (axes[1], (b1, b2), "model-b", "#FF5722"),
]:
    ax.scatter(v1.values, v2.values, s=100, color=color, zorder=5)
    
    # Label each point with the task name
    for task, x, y in zip(v1.index, v1.values, v2.values):
        ax.annotate(task, (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
    
    # y=x diagonal: points above it mean v2 > v1
    lim = [0.55, 1.0]
    ax.plot(lim, lim, "k--", linewidth=1, alpha=0.5, label="y = x (no change)")
    ax.set_title(f"{label}: v1 vs v2 scores")
    ax.set_xlabel("v1 score")
    ax.set_ylabel("v2 score")
    ax.set_xlim(*lim); ax.set_ylim(*lim)
    ax.legend()

plt.suptitle("Did v2 improve on every task? (above diagonal = yes)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Scatter with Color Encoding — Response Length vs. Flag Status

We can encode a third variable as color. Here we plot response length vs. a jittered index,
colored by whether the output was flagged.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# seaborn stripplot: x=model, y=response_length, hue=flagged
sns.stripplot(
    data=outputs_df, x="model", y="response_length",
    hue="flagged", palette={True: "crimson", False: "steelblue"},
    size=9, jitter=True, alpha=0.85, ax=ax
)

ax.set_title("Response Length by Model (red = flagged)", fontsize=13)
ax.set_xlabel("Model")
ax.set_ylabel("Response length (characters)")
ax.legend(title="Flagged", labels=["Flagged", "Clean"])
plt.tight_layout()
plt.show()

print("\nMean lengths:")
print(outputs_df.groupby("flagged")["response_length"].mean().round(1))

---
## Your Turn — Exercise 1: Histogram of Scores per Model

1. Compute `score_stats` — a DataFrame with columns `mean`, `std`, `min`, `max` grouped by model.
2. Store the standard deviation for `model-a-v2` in `a2_std`, rounded to **3 decimal places**.
3. Create a figure with a seaborn histogram of scores, colored by model (`hue="model"`).

> **Hint:** `sns.histplot(data=df, x="score", hue="model", bins=6, ax=ax)`

In [ ]:
# YOUR CODE HERE
score_stats = None   # groupby("model")["score"].agg(["mean","std","min","max"])
a2_std      = None   # std for model-a-v2, rounded to 3 decimal places

fig, ax = None, None  # plt.subplots(figsize=(9, 5))
# sns.histplot(...)
# plt.tight_layout(); plt.show()

In [ ]:
check_type(score_stats, pd.DataFrame, "score_stats is a DataFrame")
check_contains(list(score_stats.columns), "std", "score_stats has std column")
check_approx(a2_std, 0.059, 1e-3, "a2_std (sample std from pandas)")
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")

---
## Your Turn — Exercise 2: Box Plot Comparison

Create a seaborn box plot comparing score distributions, **with tasks on the x-axis** (not models).

1. Create `fig, ax = plt.subplots(figsize=(10, 5))`.
2. Draw `sns.boxplot(data=df, x="task", y="score", ...)` on `ax`.
3. Add `sns.stripplot` on top to show individual points.
4. Rotate x-axis tick labels by 30 degrees with `ax.tick_params(axis="x", rotation=30)`.
5. Store the task name with the **lowest median score** in `hardest_task`.

> **Hint:** `df.groupby("task")["score"].median().idxmin()`

In [ ]:
# YOUR CODE HERE
hardest_task = None   # task with lowest median score

fig, ax = None, None  # plt.subplots(figsize=(10, 5))
# sns.boxplot(...)
# sns.stripplot(...)
# ax.tick_params(...)
# plt.tight_layout(); plt.show()

In [ ]:
check_equal(hardest_task, "creative_writing", "hardest task by median is creative_writing")
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")

---
## Your Turn — Exercise 3: Scatter Plot — Flagged vs. Response Length

Using `outputs_df`:
1. Compute `mean_flagged_length` — mean response length for flagged outputs, rounded to 1 dp.
2. Compute `mean_clean_length` — mean response length for non-flagged outputs, rounded to 1 dp.
3. Create a scatter plot with `response_length` on x, colored by `flagged` status.
   Use `outputs_df.index` for the y-axis (or any continuous y variable).

In [ ]:
# YOUR CODE HERE
mean_flagged_length = None   # mean response length for flagged outputs, rounded to 1 dp
mean_clean_length   = None   # mean response length for non-flagged, rounded to 1 dp

fig, ax = None, None
# scatter plot here...

In [ ]:
check_approx(mean_flagged_length, 64.7, 0.2, "mean_flagged_length")
check_approx(mean_clean_length, 94.5, 0.2, "mean_clean_length")
check_equal(mean_flagged_length < mean_clean_length, True, "flagged responses are shorter")

---
## Why This Matters for AI Research Engineering

**Histograms** tell you whether your evaluation scores are well-distributed or bimodal (a bimodal score distribution often signals two distinct response strategies in the model).

**Box plots** per model are the standard way to report score variability — not just the mean, but "how consistently does this model perform?" A model with high mean but high variance is less reliable than one with slightly lower mean and low variance.

**Scatter plots** with `hue=flagged` let you spot behavioral signatures: in this dataset, flagged outputs cluster at shorter response lengths. That's a pattern you'd want to flag for further investigation — does the model shortcut its safety reasoning when it produces harmful content?

The v1-vs-v2 scatter is the standard "did the new version improve?" chart in model evaluation. Points above the diagonal = improvement; below = regression. A single chart tells the story faster than a table of numbers.

## Summary

| What | Code |
|------|------|
| Histogram (matplotlib) | `ax.hist(values, bins=n)` |
| Histogram (seaborn) | `sns.histplot(data=df, x="col", kde=True, ax=ax)` |
| Box plot | `sns.boxplot(data=df, x="group", y="val", ax=ax)` |
| Individual points | `sns.stripplot(data=df, x="group", y="val", ax=ax)` |
| Scatter | `ax.scatter(x, y, s=size, color=color)` |
| Color-encoded scatter | `sns.stripplot(..., hue="col", palette={...})` |
| Set theme | `sns.set_theme(style="whitegrid")` |

**Next:** Notebook 3 — heatmaps and multi-panel figures.